# Contexte
Une entreprise possède plusieurs bâtiments équipés de capteurs IoT.
Chaque capteur collecte régulièrement des informations sur la température, l'humidité, la pression,
la consommation énergétique, le bâtiment, la date et l'heure de la mesure.
Chaque mesure possède également un état (OK, ALERTE et ERREUR).
L'objectif de l'atelier est de construire un modèle capable de prédire automatiquement l'état d'un
capteur à partir de ses mesures.
L'atelier suivra le workflow classique du Machine Learning :
Dataset → Chargement → Exploration → Nettoyage → X / y → Train / Test → Prétraitement →
Modèle → fit()→ predict()→ Évaluation → Sauvegarde → Chargement → Réutilisation

In [1]:
# ! pip install seaborn matplotlib pandas scikit-learn joblib

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
df = pd.read_csv("../data/mesures_capteurs.csv")
df.head()

,id_mesure,date_heure,id_capteur,batiment,temperature,humidite,pression,consommation,etat
0,M0413,2026-01-22 04:00:00,C005,B002,25.46,58.06,1008.95,287.28,OK
1,M0290,2026-01-17 01:00:00,C002,B001,24.00,79.73,993.39,116.20,OK
2,M0077,2026-01-08 04:00:00,C005,B002,25.82,54.47,1010.32,288.50,OK
3,M0079,2026-01-08 06:00:00,C007,B003,28.23,69.39,1019.62,136.65,OK
4,M0183,2026-01-12 14:00:00,C003,B001,20.58,53.80,1016.58,182.62,OK


# Partie 1 – Gestion des doublons

## 1) vérifier l’existence de doublons dans df

In [4]:
nb_doublons = df.duplicated().sum()
print(f"Nombre de doublons : {nb_doublons}")

Nombre de doublons : 5


## 2) le cas échéant, supprimer les doublons puis vérifier la suppression

In [5]:
# 2) Suppression des doublons (le cas échéant) puis vérification
if nb_doublons > 0:
    df = df.drop_duplicates().reset_index(drop=True)

print(f"Nombre de doublons après suppression : {df.duplicated().sum()}")
print("Nouvelles dimensions :", df.shape)


Nombre de doublons après suppression : 0
Nouvelles dimensions : (600, 9)


# Partie 2 – Sélection de y (cible) et X (caractéristiques)

## 1) Définir "etat" comme la cible ou valeur à prédire et "temperature", "humidite", "pression" et "consommation" comme caractéristiques ou variables explicatives

In [7]:
print("Valeurs manquantes de la cible 'etat' :", df["etat"].isna().sum())
df = df.dropna(subset=["etat"]).reset_index(drop=True)
print("Dimensions après retrait des cibles manquantes :", df.shape)


Valeurs manquantes de la cible 'etat' : 0
Dimensions après retrait des cibles manquantes : (596, 9)


In [8]:
features = ["temperature", "humidite", "pression", "consommation"]
target = "etat"

X = df[features]
y = df[target]

## 2) Afficher les cinq premières lignes de X et de y

In [13]:
X.head()

,temperature,humidite,pression,consommation
0,25.46,58.06,1008.95,287.28
1,24.00,79.73,993.39,116.20
2,25.82,54.47,1010.32,288.50
3,28.23,69.39,1019.62,136.65
4,20.58,53.80,1016.58,182.62


In [16]:
y.head()

0    OK
1    OK
2    OK
3    OK
4    OK
Name: etat, dtype: str

## 3) Quel est le type du problème de machine learning ?

Il s'agit d'un problème d'**apprentissage supervisé de classification multi-classes** : la cible
etat est une variable catégorielle à trois modalités (OK, ALERTE, ERREUR), et le modèle
doit apprendre, à partir d'exemples déjà étiquetés, à prédire la bonne classe pour de nouvelles
observations.